# Glyph — Fully Connected RAG Evaluation Notebook

This notebook evaluates the **actual Glyph RAG pipeline** from `backend/rag.py`.

It directly uses Glyph's production functions:

- `get_vector_store(...)`
- `get_conversation_chain(...)`

Therefore, the evaluation path is:

**Evaluation PDF → Glyph PDF chunking → BAAI/bge-small-en-v1.5 embeddings → FAISS → MMR retrieval (`k=6`, `fetch_k=20`) → ConversationBufferMemory → Groq Llama model → generated answer**

This notebook measures the metrics required by Section 9:

1. Response Accuracy
2. Retrieval Relevance
3. Average Response Latency
4. User Satisfaction

It also:
- records retrieved source documents,
- records page/chunk metadata,
- preserves generated answers,
- supports manual evaluation,
- compares measured results with the Section 9 reported values,
- exports CSV/JSON results.

**Important:** The notebook does not fabricate scores. Human evaluation fields must be completed after reviewing the actual outputs.

## 1. Before You Run the Notebook

From the root of the Glyph repository, install the backend dependencies:

```bash
cd backend
pip install -r requirements.txt
```

Make sure your `backend/.env` contains:

```env
GROQ_API_KEY=your_key
GROQ_MODEL_NAME=llama-3.3-70b-versatile
```

The notebook needs access to the same environment because Glyph's `rag.py` loads these values with `python-dotenv`.

You also need an evaluation dataset and PDFs.

Recommended layout:

```text
Glyph/
├── backend/
│   ├── main.py
│   ├── rag.py
│   └── .env
├── frontend/
├── notebooks/
│   └── evaluation.ipynb
└── evaluation_data/
    ├── pdfs/
    │   ├── document1.pdf
    │   └── document2.pdf
    └── evaluation_questions.csv
```

The evaluation CSV should contain:

- `id`
- `pdf`
- `question`
- `reference_answer`
- `relevant_pages`
- `difficulty`
- `satisfaction`

`reference_answer` is the human-written correct answer used for accuracy evaluation.

In [12]:
# 2. Imports and paths

from pathlib import Path
import sys
import os
import json
import time
import re
import uuid

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Notebook location is expected to be:
# Glyph/notebooks/evaluation.ipynb
NOTEBOOK_DIR = Path.cwd()

# If running from the Glyph root, this will point to the same place.
if (NOTEBOOK_DIR.name == "notebooks"):
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

BACKEND_DIR = PROJECT_ROOT / "backend"
EVAL_DIR = PROJECT_ROOT / "evaluation_data"
PDF_DIR = EVAL_DIR / "pdfs"
RESULTS_DIR = PROJECT_ROOT / "evaluation_results"

PDF_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

backend_path = str(BACKEND_DIR.resolve())
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

print("Project root:", PROJECT_ROOT.resolve())
print("Backend:", BACKEND_DIR.resolve())
print("Evaluation PDFs:", PDF_DIR.resolve())
print("Results:", RESULTS_DIR.resolve())

Project root: E:\kajal\Engg\Glyph
Backend: E:\kajal\Engg\Glyph\backend
Evaluation PDFs: E:\kajal\Engg\Glyph\evaluation_data\pdfs
Results: E:\kajal\Engg\Glyph\evaluation_results


In [13]:
# 3. Import Glyph's actual production RAG functions

# Load the backend .env explicitly so rag.py can read Groq settings when the
# notebook is executed from the notebooks directory.
from dotenv import load_dotenv
dotenv_path = BACKEND_DIR / ".env"
if dotenv_path.exists():
    load_dotenv(dotenv_path)

from rag import (
    get_pdf_text,
    get_vector_store,
    get_conversation_chain,
)

print("Successfully imported Glyph's production RAG pipeline.")
print("Using get_vector_store:", get_vector_store)
print("Using get_conversation_chain:", get_conversation_chain)


Successfully imported Glyph's production RAG pipeline.
Using get_vector_store: <function get_vector_store at 0x0000019710F14880>
Using get_conversation_chain: <function get_conversation_chain at 0x0000019710F6CD50>


## 4. Evaluation Dataset

Create `evaluation_data/evaluation_questions.csv`.

Example:

```csv
id,pdf,question,reference_answer,relevant_pages,difficulty,satisfaction
Q01,paper.pdf,What is the main objective?,The main objective is ...,1,easy,
Q02,paper.pdf,What methodology is used?,The study uses ...,3,medium,
```

For the final capstone evaluation, use **real questions and reference answers based on your actual test PDFs**.

Do not use the example values as evaluation data.

In [17]:
EVAL_CSV = EVAL_DIR / "evaluation_questions.csv"

if not EVAL_CSV.exists():
    template = pd.DataFrame([
        {
            "id": "Q01",
            "pdf": "REPLACE_WITH_REAL_PDF",
            "question": "Replace with a real question from your PDF.",
            "reference_answer": "Replace with the correct answer supported by the PDF.",
            "relevant_pages": "1",
            "difficulty": "easy",
            "satisfaction": ""
        }
    ])
    template.to_csv(EVAL_CSV, index=False)
    print("Created template:", EVAL_CSV)
    print("Edit this file, then rerun this cell.")
else:
    print("Found evaluation dataset:", EVAL_CSV)

eval_df = pd.read_csv(EVAL_CSV)

required_columns = ["id", "pdf", "question", "reference_answer"]
missing = [c for c in required_columns if c not in eval_df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

eval_df.head()

Found evaluation dataset: e:\kajal\Engg\Glyph\evaluation_data\evaluation_questions.csv


,id,pdf,question,reference_answer,relevant_pages,difficulty,satisfaction
0,Q01,REPLACE_WITH_REAL_PDF.pdf,Replace with a real question from your PDF.,Replace with the correct answer supported by t...,1,easy,NaN


In [18]:
# 5. Validate that all referenced PDFs exist

missing_pdfs = []

for pdf_name in eval_df["pdf"].dropna().unique():
    pdf_path = PDF_DIR / str(pdf_name)
    if not pdf_path.exists():
        missing_pdfs.append(str(pdf_path))

if missing_pdfs:
    print("Missing PDFs:")
    for p in missing_pdfs:
        print(" -", p)
    print("\nPlace the PDFs in evaluation_data/pdfs/ and rerun this cell.")
else:
    print("All evaluation PDFs found.")

Missing PDFs:
 - e:\kajal\Engg\Glyph\evaluation_data\pdfs\REPLACE_WITH_REAL_PDF.pdf

Place the PDFs in evaluation_data/pdfs/ and rerun this cell.


## 6. Build Glyph's Actual FAISS Vector Store

This cell uses **the exact `get_vector_store()` implementation from Glyph's `backend/rag.py`**.

That means the evaluation uses the production settings currently in the repository:

- RecursiveCharacterTextSplitter
- chunk size = 1400
- overlap = 250
- `BAAI/bge-small-en-v1.5`
- FAISS
- page/chunk metadata

The PDFs are grouped into a single vector store, matching the multi-PDF upload behavior of Glyph.

In [20]:
# Build file-like objects in the same form expected by Glyph's get_vector_store().

class EvaluationPDFFile:
    def __init__(self, path):
        self.path = Path(path)
        self.name = self.path.name
        self.filename = self.path.name
        self._file = open(self.path, "rb")

    def seek(self, *args):
        return self._file.seek(*args)

    def read(self, *args):
        return self._file.read(*args)

    def close(self):
        self._file.close()


pdf_paths = [PDF_DIR / str(name) for name in eval_df["pdf"].dropna().unique()]

missing_paths = [p for p in pdf_paths if not p.exists()]
if missing_paths:
    missing_list = "\n".join(str(p) for p in missing_paths)
    raise FileNotFoundError(
        "One or more evaluation PDFs are missing. "
        f"Place the required files under {PDF_DIR.resolve()} and rerun cell 5.\n"
        "Missing files:\n" + missing_list
    )

pdf_files = [EvaluationPDFFile(p) for p in pdf_paths]

print("PDFs used for evaluation:")
for p in pdf_paths:
    print(" -", p.name)

vector_store, chunk_count = get_vector_store(pdf_files)

print(f"Glyph created FAISS vector store with {chunk_count} chunks.")


FileNotFoundError: One or more evaluation PDFs are missing. Place the required files under E:\kajal\Engg\Glyph\evaluation_data\pdfs and rerun cell 5.
Missing files:
e:\kajal\Engg\Glyph\evaluation_data\pdfs\REPLACE_WITH_REAL_PDF.pdf

## 7. Create Glyph's Actual Conversational RAG Chain

This uses the same `get_conversation_chain()` function as the FastAPI application.

Therefore the chain uses:

- Groq-hosted Llama model configured in `.env`
- temperature = 0
- ConversationBufferMemory
- the grounding prompt from `rag.py`
- MMR retrieval
- `k=6`
- `fetch_k=20`
- `return_source_documents=True`

This is the actual production chain, not a separate evaluation-only implementation.

In [ ]:
conversation_chain = get_conversation_chain(vector_store)

print("Glyph ConversationalRetrievalChain created successfully.")
print("LLM and retriever are configured by backend/rag.py.")

## 8. Run One Test Question

Before running the full evaluation, test one question.

This cell also demonstrates how the notebook accesses the retrieved source documents.

In [ ]:
test_question = str(eval_df.iloc[0]["question"])

# Ask the same chain used by Glyph.
start = time.perf_counter()
test_response = conversation_chain.invoke({"question": test_question})
test_latency = time.perf_counter() - start

print("QUESTION:")
print(test_question)

print("\nANSWER:")
print(test_response.get("answer", ""))

print("\nLATENCY:")
print(f"{test_latency:.3f} seconds")

print("\nRETRIEVED SOURCES:")
for i, doc in enumerate(test_response.get("source_documents", []), start=1):
    print(
        f"{i}. "
        f"document={doc.metadata.get('document')} | "
        f"page={doc.metadata.get('page')} | "
        f"chunk={doc.metadata.get('chunk')}"
    )

## 9. Run the Full Evaluation

Each question is sent to the real Glyph chain.

For every question, the notebook records:

- generated answer,
- latency,
- retrieved document count,
- retrieved document names,
- retrieved page numbers,
- retrieved chunk IDs.

A fresh chain is **not** created for every question because Glyph is conversational and uses `ConversationBufferMemory`. Questions belonging to the same evaluation session should therefore be ordered intentionally.

For independent question evaluation, the notebook uses a fresh chain for each question. This prevents one test question's chat history from affecting another question's answer.

This makes the accuracy/retrieval evaluation cleaner and avoids cross-question contamination.

In [ ]:
def make_fresh_glyph_chain():
    # Reuse the same vector store, but create fresh memory.
    return get_conversation_chain(vector_store)


def run_single_evaluation(row):
    chain = make_fresh_glyph_chain()

    question = str(row["question"])

    start = time.perf_counter()
    response = chain.invoke({"question": question})
    latency = time.perf_counter() - start

    answer = response.get("answer", "")
    source_docs = response.get("source_documents", [])

    source_records = []
    for doc in source_docs:
        source_records.append({
            "document": doc.metadata.get("document"),
            "page": doc.metadata.get("page"),
            "chunk": doc.metadata.get("chunk"),
        })

    return {
        "id": row["id"],
        "pdf": row["pdf"],
        "question": question,
        "reference_answer": row["reference_answer"],
        "generated_answer": answer,
        "latency_sec": latency,
        "retrieved_count": len(source_docs),
        "retrieved_sources": json.dumps(source_records),
        "retrieved_documents": ", ".join(
            sorted(set(str(x["document"]) for x in source_records if x["document"] is not None))
        ),
        "retrieved_pages": ", ".join(
            sorted(set(str(x["page"]) for x in source_records if x["page"] is not None))
        ),
        "retrieved_chunks": ", ".join(
            sorted(set(str(x["chunk"]) for x in source_records if x["chunk"] is not None))
        ),
        "correct_binary": "",
        "partial_score": "",
        "retrieval_relevant": "",
        "satisfaction": row.get("satisfaction", ""),
        "failure_case": "",
    }


evaluation_outputs = []

for idx, row in eval_df.iterrows():
    print(f"Running {idx + 1}/{len(eval_df)}: {row['id']}")

    try:
        result = run_single_evaluation(row)
        evaluation_outputs.append(result)

        print(f"  latency: {result['latency_sec']:.3f}s")
        print(f"  retrieved: {result['retrieved_count']} chunks")
    except Exception as e:
        print("  ERROR:", repr(e))
        evaluation_outputs.append({
            "id": row["id"],
            "pdf": row["pdf"],
            "question": row["question"],
            "reference_answer": row["reference_answer"],
            "generated_answer": "",
            "latency_sec": np.nan,
            "retrieved_count": 0,
            "retrieved_sources": "[]",
            "retrieved_documents": "",
            "retrieved_pages": "",
            "retrieved_chunks": "",
            "correct_binary": "",
            "partial_score": "",
            "retrieval_relevant": "",
            "satisfaction": row.get("satisfaction", ""),
            "failure_case": f"Pipeline error: {repr(e)}",
        })

results_df = pd.DataFrame(evaluation_outputs)

print("\nEvaluation run complete.")
results_df[[
    "id", "question", "generated_answer", "latency_sec",
    "retrieved_count", "retrieved_documents", "retrieved_pages"
]]

## 10. Manual Evaluation

Now inspect each generated answer against the reference answer.

Fill in:

### `correct_binary`
- `1` = correct and sufficiently complete
- `0` = incorrect

### `partial_score`
- `1` = fully correct
- `0.5` = partially correct or incomplete
- `0` = incorrect

### `retrieval_relevant`
- `1` = retrieved chunks contain relevant evidence needed to answer
- `0` = retrieved chunks are not relevant

### `satisfaction`
- 1 to 5 based on human evaluator judgment

### `failure_case`
Describe notable failures.

You can save the generated results first, open the CSV in Excel, enter the scores, save it, then reload it in the notebook.

In [ ]:
RAW_RESULTS_CSV = RESULTS_DIR / "evaluation_results_raw.csv"

results_df.to_csv(RAW_RESULTS_CSV, index=False)

print("Raw evaluation results saved to:")
print(RAW_RESULTS_CSV)
print("\nOpen this CSV and fill the manual evaluation columns:")
print("- correct_binary")
print("- partial_score")
print("- retrieval_relevant")
print("- satisfaction")
print("- failure_case")

In [ ]:
# Reload manually scored results after editing the CSV.

SCORED_RESULTS_CSV = RESULTS_DIR / "evaluation_results_scored.csv"

if SCORED_RESULTS_CSV.exists():
    scored_df = pd.read_csv(SCORED_RESULTS_CSV)
    print("Loaded scored results:", SCORED_RESULTS_CSV)
else:
    print("No scored CSV found yet.")
    print("For convenience, the raw results are currently loaded.")
    scored_df = results_df.copy()

scored_df.head()

## 11. Calculate the Four Section 9 Metrics

In [ ]:
def numeric_values(series):
    return pd.to_numeric(series, errors="coerce").dropna()


accuracy_binary = numeric_values(scored_df["correct_binary"])
partial_scores = numeric_values(scored_df["partial_score"])
retrieval_scores = numeric_values(scored_df["retrieval_relevant"])
latencies = numeric_values(scored_df["latency_sec"])
satisfaction_scores = numeric_values(scored_df["satisfaction"])

response_accuracy_pct = (
    accuracy_binary.mean() * 100 if len(accuracy_binary) else np.nan
)

response_accuracy_partial_pct = (
    partial_scores.mean() * 100 if len(partial_scores) else np.nan
)

retrieval_relevance_pct = (
    retrieval_scores.mean() * 100 if len(retrieval_scores) else np.nan
)

average_latency_sec = (
    latencies.mean() if len(latencies) else np.nan
)

user_satisfaction_5 = (
    satisfaction_scores.mean() if len(satisfaction_scores) else np.nan
)

metrics = pd.DataFrame({
    "Metric": [
        "Response Accuracy",
        "Response Accuracy (Partial Credit)",
        "Retrieval Relevance",
        "Average Response Latency",
        "User Satisfaction"
    ],
    "Measured Value": [
        f"{response_accuracy_pct:.2f}%" if not np.isnan(response_accuracy_pct) else "N/A",
        f"{response_accuracy_partial_pct:.2f}%" if not np.isnan(response_accuracy_partial_pct) else "N/A",
        f"{retrieval_relevance_pct:.2f}%" if not np.isnan(retrieval_relevance_pct) else "N/A",
        f"{average_latency_sec:.2f} sec" if not np.isnan(average_latency_sec) else "N/A",
        f"{user_satisfaction_5:.2f}/5" if not np.isnan(user_satisfaction_5) else "N/A",
    ]
})

metrics

## 12. Section 9 Reported Values vs. Notebook Measurements

The capstone report currently states:

- Response Accuracy: 93%
- Retrieval Relevance: 100%
- Average Latency: 3.15 seconds
- User Satisfaction: 4.7/5

These are shown here as **reported values**.

The notebook measurements are calculated from the actual evaluation run above.

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Response Accuracy",
        "Retrieval Relevance",
        "Average Latency",
        "User Satisfaction"
    ],
    "Target / Baseline": [
        "85%",
        "High",
        "<5 seconds",
        ">4/5"
    ],
    "Section 9 Reported": [
        "93%",
        "100%",
        "3.15 sec",
        "4.7/5"
    ],
    "Notebook Measured": [
        f"{response_accuracy_pct:.2f}%" if not np.isnan(response_accuracy_pct) else "N/A",
        f"{retrieval_relevance_pct:.2f}%" if not np.isnan(retrieval_relevance_pct) else "N/A",
        f"{average_latency_sec:.2f} sec" if not np.isnan(average_latency_sec) else "N/A",
        f"{user_satisfaction_5:.2f}/5" if not np.isnan(user_satisfaction_5) else "N/A",
    ]
})

comparison

## 13. Visualizations

In [ ]:
# Response Accuracy
if not np.isnan(response_accuracy_pct):
    plt.figure(figsize=(7, 4))
    plt.bar(
        ["Target", "Notebook", "Section 9"],
        [85, response_accuracy_pct, 93]
    )
    plt.ylabel("Accuracy (%)")
    plt.title("Response Accuracy")
    plt.ylim(0, 100)
    plt.show()

# Retrieval Relevance
if not np.isnan(retrieval_relevance_pct):
    plt.figure(figsize=(7, 4))
    plt.bar(
        ["Notebook", "Section 9"],
        [retrieval_relevance_pct, 100]
    )
    plt.ylabel("Relevance (%)")
    plt.title("Retrieval Relevance")
    plt.ylim(0, 100)
    plt.show()

# Latency
if not np.isnan(average_latency_sec):
    plt.figure(figsize=(7, 4))
    plt.bar(
        ["Target Limit", "Notebook", "Section 9"],
        [5, average_latency_sec, 3.15]
    )
    plt.ylabel("Seconds")
    plt.title("Average Response Latency")
    plt.show()

# Satisfaction
if not np.isnan(user_satisfaction_5):
    plt.figure(figsize=(7, 4))
    plt.bar(
        ["Target", "Notebook", "Section 9"],
        [4, user_satisfaction_5, 4.7]
    )
    plt.ylabel("Rating (1–5)")
    plt.title("User Satisfaction")
    plt.ylim(0, 5)
    plt.show()

In [ ]:
# Latency distribution

if len(latencies):
    plt.figure(figsize=(7, 4))
    plt.hist(latencies, bins=min(10, max(1, len(latencies))))
    plt.xlabel("Latency (seconds)")
    plt.ylabel("Number of Questions")
    plt.title("Response Latency Distribution")
    plt.show()

## 14. Failure Analysis

Use the `failure_case` field to document failures observed during evaluation.

The documentation identifies two important failure modes:

1. Scanned/image-only PDFs may produce incomplete extraction.
2. Information spread across distant sections may be missed by top-ranked retrieval.

The notebook also records any runtime failures encountered during evaluation.

In [ ]:
failure_mask = (
    scored_df["failure_case"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

if failure_mask.any():
    display(
        scored_df.loc[
            failure_mask,
            ["id", "pdf", "question", "failure_case"]
        ]
    )
else:
    print("No failure cases recorded.")

## 15. Export Final Evaluation Artifacts

These files can be committed to the repository if appropriate:

- `evaluation_results_raw.csv`
- `evaluation_results_scored.csv`
- `evaluation_summary.csv`
- `evaluation_summary.json`

The raw file preserves the direct output of the real Glyph RAG pipeline. The scored file contains human evaluation judgments.

In [ ]:
summary = {
    "project": "Glyph",
    "response_accuracy_pct": None if np.isnan(response_accuracy_pct) else float(response_accuracy_pct),
    "response_accuracy_partial_credit_pct": None if np.isnan(response_accuracy_partial_pct) else float(response_accuracy_partial_pct),
    "retrieval_relevance_pct": None if np.isnan(retrieval_relevance_pct) else float(retrieval_relevance_pct),
    "average_response_latency_sec": None if np.isnan(average_latency_sec) else float(average_latency_sec),
    "user_satisfaction_5": None if np.isnan(user_satisfaction_5) else float(user_satisfaction_5),
    "num_questions": int(len(scored_df)),
    "section_9_reported": {
        "response_accuracy_pct": 93.0,
        "retrieval_relevance_pct": 100.0,
        "average_latency_sec": 3.15,
        "user_satisfaction_5": 4.7,
    }
}

summary_df = pd.DataFrame({
    "Metric": [
        "Response Accuracy",
        "Retrieval Relevance",
        "Average Response Latency",
        "User Satisfaction"
    ],
    "Notebook Measured": [
        summary["response_accuracy_pct"],
        summary["retrieval_relevance_pct"],
        summary["average_response_latency_sec"],
        summary["user_satisfaction_5"]
    ],
    "Section 9 Reported": [
        93.0,
        100.0,
        3.15,
        4.7
    ]
})

summary_df.to_csv(RESULTS_DIR / "evaluation_summary.csv", index=False)

with open(RESULTS_DIR / "evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

# Save scored results as the canonical evaluation file.
scored_df.to_csv(RESULTS_DIR / "evaluation_results_scored.csv", index=False)

print("Saved evaluation artifacts to:", RESULTS_DIR.resolve())
print(summary_df)

## 16. Appendix C Submission Checklist

Before submitting this notebook:

- [ ] `evaluation.ipynb` is located in `notebooks/`
- [ ] Evaluation PDFs are available in `evaluation_data/pdfs/`
- [ ] `evaluation_questions.csv` contains real questions
- [ ] Each question has a human-written reference answer
- [ ] The notebook imports `get_vector_store` and `get_conversation_chain` from Glyph's actual `backend/rag.py`
- [ ] The notebook successfully runs the real FAISS + MMR + Groq RAG pipeline
- [ ] Generated answers are saved
- [ ] Retrieved sources/pages/chunks are saved
- [ ] Latency is measured automatically
- [ ] Accuracy is manually evaluated
- [ ] Retrieval relevance is manually evaluated
- [ ] User satisfaction ratings are collected
- [ ] Failure cases are documented
- [ ] The notebook produces the final metrics
- [ ] Section 9 values are updated to match the actual measured results if they differ

**Important:** If the measured notebook results differ from the current Section 9 numbers, update Section 9 rather than changing the notebook results.